# 08 · Importancia de variables — documentación y reporte

**Pregunta que responde.** Matriz de riesgos del Prof. Marcos Alania: *«El modelo afirma que el ROTI antecede al S4. ¿Lo aprendió la red o es una asunción suya?»*. Se pidió un análisis de importancia de variables o mapas de saliencia.

**Qué hace este notebook.** Documenta el análisis, los scripts en orden y **genera tablas y figuras sin reentrenar**.

> ⚠️ **No reentrenar desde aquí.** Las corridas se ejecutaron con los scripts de la sección 2.

| Elemento | Valor |
|---|---|
| Modelo analizado | LSTM apilada · D' · S4 + hora + ROTI · ventana 180 min · WFL α = 1.5, β = 10 |
| Datos | Validación completa sin objetivos reconstruidos (92 932 ventanas; 853 inicios de evento) |
| Métodos | Importancia por permutación agrupada (3 semillas × 3 repeticiones) y gradientes integrados (400 ventanas de inicio) |
| Umbral de alerta | 0.55 |

## 1. Pasos seguidos

| # | Paso | Script | Resultado |
|---|---|---|---|
| 1 | Entrenar D' con 3 semillas y medir la importancia por permutación agrupada | `scripts/correr_explicabilidad.py` | `tabla_importancia_permutacion.csv` |
| 2 | Gradientes integrados en ventanas de inicio de evento | mismo script | `tabla_saliencia_inicio.csv` |
| 3 | Confirmación: S4 + hora **sin ROTI** con la misma configuración (60 y 180 min) | `scripts/correr_confirmacion_C.py` | grupo `confirmacion_C` del registro |
| 4 | Reporte | **este notebook** | figuras y tablas |

### Métodos

- **Importancia por permutación agrupada.** Para cada grupo de variables se desordenan sus valores entre ventanas (la distribución se conserva, la relación con el evento se rompe) y se mide la caída del HSS, de la precisión y de los inicios anticipados. El seno y el coseno de cada variable cíclica se desordenan **juntos**, para no crear horas imposibles. Si al desordenar un grupo el modelo no empeora, la red no lo usa.
- **Gradientes integrados** (Sundararajan et al., 2017). Atribuyen a cada variable y a cada minuto de la ventana su contribución al pronóstico. Se aplican a las ventanas de **inicio** de evento, que es donde un precursor debería actuar.

## 2. Scripts empleados

```bash
nohup python scripts/correr_explicabilidad.py > experiments/log_explicabilidad.txt 2>&1 &
nohup python scripts/correr_confirmacion_C.py > experiments/log_confirmacion_C.txt 2>&1 &
```

```python
# correr_explicabilidad.py (extracto)
GRUPOS = {"S4": ["S4"], "Hora del día": ["daily_sin", "daily_cos"],
          "Día del año": ["day_of_year_sin", "day_of_year_cos"], "ROTI": ["ROTI"]}
for semilla, modelo, _ in modelos:
    base = metricas(predecir(modelo, X))
    for g, cols in GRUPOS.items():
        for rep in range(3):
            Xp = X.copy(); perm = rng.permutation(len(X))
            Xp[:, :, idx_de(cols)] = X[perm][:, :, idx_de(cols)]     # mismo orden para todo el grupo
            caida = base["HSS"] - metricas(predecir(modelo, Xp))["HSS"]
at = explain.gradientes_integrados(modelo42, X[ventanas_de_inicio], pasos=32)

# correr_confirmacion_C.py
for lb in [180, 60]:
    correr_variante(variante(base, **{"ventanas.lookback": lb}), datos, ft.CONJUNTOS["B_S4_ciclica"],
                    f"C · LB{lb} · b10", "confirmacion_C", barrido_umbral=True)
```

## 3. Reporte (no entrena)

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, ruta
from s4.registry import leer, resumir_semillas
cfg = cargar_config("../config/ablacion_2025_2026.yaml")
REG = ruta(cfg, "experimentos") / f"registro_{cfg['nombre']}.csv"
FIG, TAB = ruta(cfg, "reportes") / "figures", ruta(cfg, "reportes") / "tables"
AZUL, CELESTE, GRIS, AMBAR, ROJO = "#0000AF", "#0099FF", "#B8BED6", "#C77C02", "#C62828"

### 3.1 Importancia por permutación

In [ ]:
imp = pd.read_csv(TAB / "tabla_importancia_permutacion.csv")
display(imp.round(4))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, (m, tit) in zip(axes, [("caida_HSS", "Caída del HSS al desordenar el grupo"), ("caida_POD_inicio", "Caída de los inicios anticipados")]):
    t = imp.sort_values(f"{m}_mean")
    colores = [ROJO if g == "ROTI" else (AZUL if g in ("S4", "Hora del día") else GRIS) for g in t["grupo"]]
    ax.barh(t["grupo"], t[f"{m}_mean"], xerr=t[f"{m}_std"], color=colores, capsize=4)
    ax.axvline(0, color="black", lw=0.8); ax.set_title(tit); ax.spines[["top", "right"]].set_visible(False)
plt.suptitle("Importancia por permutación agrupada (3 semillas × 3 repeticiones, validación)")
plt.tight_layout(); plt.savefig(FIG / "fig_importancia_permutacion.png", dpi=200); plt.show()

### 3.2 Gradientes integrados en ventanas de inicio de evento

In [ ]:
sal = pd.read_csv(TAB / "tabla_saliencia_inicio.csv")
grupos = {"S4": ["S4"], "Hora del día": ["daily_sin", "daily_cos"], "Día del año": ["day_of_year_sin", "day_of_year_cos"], "ROTI": ["ROTI"]}
G = pd.DataFrame({g: sal[c].sum(axis=1) for g, c in grupos.items()})
part = (100 * G.sum() / G.sum().sum()).round(1)
print("Participación de cada grupo en las decisiones (%):"); print(part.to_string())
x = -sal["minutos_antes_de_t"]
fig, ax = plt.subplots(figsize=(12, 4.2))
for g, c in zip(G.columns, [AZUL, CELESTE, GRIS, ROJO]):
    ax.plot(x, G[g] / G.values.sum() * 100, label=f"{g} ({part[g]:.1f} %)", color=c, lw=2)
ax.set_xlabel("Minutos antes del instante de pronóstico (0 = ahora)"); ax.set_ylabel("Atribución media (% del total por minuto)")
ax.set_title("Gradientes integrados: la red mira sobre todo los últimos minutos de S4 y la hora; ROTI casi no interviene")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.savefig(FIG / "fig_saliencia_inicio.png", dpi=200); plt.show()
roti = sal["ROTI"].values[::-1]          # 0 = minuto actual
tramos = {"últimos 30 min": roti[:30].sum(), "30–60 min antes": roti[30:60].sum(), "1–2 h antes": roti[60:120].sum(), "2–3 h antes": roti[120:].sum()}
print("\nROTI: dónde se concentra su aporte (%)"); print((pd.Series(tramos) / roti.sum() * 100).round(1).to_string())

### 3.3 Confirmación: el modelo con y sin ROTI

In [ ]:
partes = [leer(REG, grupo="explicabilidad_D", particion="val").assign(modelo="D' · S4 + hora + ROTI · 180 min")]
try:
    partes.insert(0, leer(REG, grupo="confirmacion_C", particion="val"))
except Exception as e:
    print("Sin corridas de confirmación en el registro:", e)
conf = resumir_semillas(pd.concat(partes), por=("modelo",), metricas=("HSS_opt", "umbral_opt", "Precision_opt", "POD_inicio_opt", "RMSE_global"))
conf.to_csv(TAB / "tabla_confirmacion_C.csv", index=False)
conf.round(3)

## 4. Resultados obtenidos (validación)

### Importancia por permutación (media de 3 semillas × 3 repeticiones)

| Grupo desordenado | Caída de HSS | Caída de precisión | Caída de inicios anticipados | Aumento del RMSE en eventos |
|---|---|---|---|---|
| **S4** | **0.648 ± 0.010** | 0.603 | 61 puntos | 0.403 |
| **Hora del día** | **0.097 ± 0.012** | −0.051 | **26 puntos** | 0.071 |
| Día del año | 0.009 ± 0.013 | −0.016 | 5 puntos | 0.003 |
| **ROTI** | **−0.007 ± 0.004** | −0.004 | −1 punto | −0.001 |

### Gradientes integrados (ventanas de inicio)

| Grupo | Participación en las decisiones |
|---|---|
| S4 | 48.2 % |
| Hora del día | 38.4 % |
| Día del año | 11.0 % |
| ROTI | 2.5 % (84.6 % de ese aporte en los últimos 30 min; 0 % entre 2 y 3 h antes) |

### Confirmación (α = 1.5, β = 10, umbral calibrado, 3 semillas)

| Modelo | HSS calibrado | Umbral óptimo | Precisión | Inicios anticipados | RMSE global |
|---|---|---|---|---|---|
| **C · S4 + hora · 60 min** | **0.658 ± 0.003** | **0.55 (las 3 semillas)** | 65 % | 59 % | **0.057** |
| C · S4 + hora · 180 min | 0.655 ± 0.011 | 0.55 | 65 % | 58 % | 0.059 |
| D' · S4 + hora + ROTI · 180 min | 0.661 ± 0.009 | 0.57 | 67 % | 57 % | 0.060 |

## 5. Conclusiones

1. **La red no usa ROTI.** Desordenarlo no reduce el HSS (−0.007 ± 0.004: incluso mejora levemente en las 3 semillas), y explica solo el 2.5 % de las decisiones, concentrado en los últimos 30 minutos y no horas antes, como haría un precursor. El modelo sin ROTI rinde igual (0.658 frente a 0.661). La mejora aparente de la ablación complementaria (HSS de 0.55 a 0.59) era variabilidad entre corridas (≈ 0.03 por el no-determinismo de la GPU).
2. **S4 es la base del pronóstico:** sin su historia el modelo colapsa (HSS ≈ 0.01).
3. **La hora del día explica la anticipación:** al desordenarla se pierden 26 puntos de inicios anticipados. La red aprendió la climatología del centelleo ecuatorial: las burbujas de plasma se forman después de la puesta del Sol. Combinando la hora con la evolución reciente del S4, alerta antes de que el índice cruce el umbral.
4. **El día del año tiene un efecto menor** (efecto estacional de los equinoccios).
5. **Respuesta al especialista:** la red **no** aprendió que ROTI antecede al S4; la anticipación proviene de la dinámica reciente del S4 y de la hora del día. Debe retirarse la afirmación de que TEC o ROTI son precursores aprendidos por el modelo (incluido el ROC-AUC de 0.753 del artículo anterior, obtenido con datos que filtraban información del futuro).
6. **Consecuencia para el modelo final:** se elige **C · S4 + hora · 60 min**: igual desempeño, la menor variabilidad (± 0.003), umbral estable (0.55) y, sobre todo, **no depende de que ROTI esté disponible en tiempo real**.

**Limitaciones.** El análisis se hizo en Jicamarca con ROTI derivado de un receptor GNSS y con su cadencia nativa (≈ 1 de cada 3 minutos con medición real). Precursores con mayor resolución o de otras estaciones podrían comportarse distinto.